In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Set clean styling for visualization export
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'Helvetica'

In [6]:
# 1. Load data
df_2019 = pd.read_csv("Divvy_Trips_2019_Q1.csv")
df_2020 = pd.read_csv("Divvy_Trips_2020_Q1.csv")

df_2019.head()

,trip_id,start_time,end_time,bikeid,tripduration,from_station_id,from_station_name,to_station_id,to_station_name,usertype,gender,birthyear
0,21742443,2019-01-01 00:04:37,2019-01-01 00:11:07,2167,390.0,199,Wabash Ave & Grand Ave,84,Milwaukee Ave & Grand Ave,Subscriber,Male,1989.0
1,21742444,2019-01-01 00:08:13,2019-01-01 00:15:34,4386,441.0,44,State St & Randolph St,624,Dearborn St & Van Buren St (*),Subscriber,Female,1990.0
2,21742445,2019-01-01 00:13:23,2019-01-01 00:27:12,1524,829.0,15,Racine Ave & 18th St,644,Western Ave & Fillmore St (*),Subscriber,Female,1994.0
3,21742446,2019-01-01 00:13:45,2019-01-01 00:43:28,252,"1,783.0",123,California Ave & Milwaukee Ave,176,Clark St & Elm St,Subscriber,Male,1993.0
4,21742447,2019-01-01 00:14:52,2019-01-01 00:20:56,1170,364.0,173,Mies van der Rohe Way & Chicago Ave,35,Streeter Dr & Grand Ave,Subscriber,Male,1994.0


In [7]:
df_2020.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,EACB19130B0CDA4A,docked_bike,2020-01-21 20:06:59,2020-01-21 20:14:30,Western Ave & Leland Ave,239,Clark St & Leland Ave,326.0,41.9665,-87.6884,41.9671,-87.6674,member
1,8FED874C809DC021,docked_bike,2020-01-30 14:22:39,2020-01-30 14:26:22,Clark St & Montrose Ave,234,Southport Ave & Irving Park Rd,318.0,41.9616,-87.6660,41.9542,-87.6644,member
2,789F3C21E472CA96,docked_bike,2020-01-09 19:29:26,2020-01-09 19:32:17,Broadway & Belmont Ave,296,Wilton Ave & Belmont Ave,117.0,41.9401,-87.6455,41.9402,-87.6530,member
3,C9A388DAC6ABF313,docked_bike,2020-01-06 16:17:07,2020-01-06 16:25:56,Clark St & Randolph St,51,Fairbanks Ct & Grand Ave,24.0,41.8846,-87.6319,41.8918,-87.6206,member
4,943BC3CBECCFD662,docked_bike,2020-01-30 08:37:16,2020-01-30 08:42:48,Clinton St & Lake St,66,Wells St & Hubbard St,212.0,41.8856,-87.6418,41.8899,-87.6343,member


In [8]:
# 2. Schema Harmonization
rename_2019 = {
    "trip_id": "ride_id",
    "start_time": "started_at",
    "end_time": "ended_at",
    "from_station_name": "start_station_name",
    "from_station_id": "start_station_id",
    "to_station_name": "end_station_name",
    "to_station_id": "end_station_id",
    "usertype": "member_casual"
}
df_2019 = df_2019.rename(columns=rename_2019)
df_2019["member_casual"] = df_2019["member_casual"].replace({
    "Subscriber": "member",
    "Customer": "casual"
})

common_cols = [
    "ride_id", "started_at", "ended_at", "start_station_name", "start_station_id",
    "end_station_name", "end_station_id", "member_casual"
]

df_2019["ride_id"] = df_2019["ride_id"].astype(str)
df_2020["ride_id"] = df_2020["ride_id"].astype(str)

df = pd.concat([df_2019[common_cols], df_2020[common_cols]], ignore_index=True)

In [9]:
# 3. Transformation and calculations
df['started_at'] = pd.to_datetime(df["started_at"])
df['ended_at'] = pd.to_datetime(df["ended_at"])
df["ride_length_min"] = (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60.0
df["day_name"] = df["started_at"].dt.day_name()
df["hour"] = df["started_at"].dt.hour

In [10]:
# 4. Outlier Removal & Data Filtering
# Remove rides shorter than 1 minute and longer than 24 hours (1440 minutes)
clean_df = df[
    (df["ride_length_min"] >= 1.0) &
    (df["ride_length_min"] >= 1440.0) &
    (df["start_station_name"] != "HQ QR")
    ].dropna(subset=["start_station_name", "end_station_name"]).copy()
    

In [12]:
# 5. Descriptive Analytics
stats = clean_df.groupby("member_casual")["ride_length_min"].agg(
    total_rides='count',
    mean_duration='mean',
    median_duration='median',
    iqr=lambda x: np.percentile(x, 75) - np.percentile(x, 25),
    std_dev='std'
).reset_index()

print("=== STATISTICAL SUMMARY ===")
print(stats.to_string(index=False))

=== STATISTICAL SUMMARY ===
member_casual  total_rides  mean_duration  median_duration         iqr      std_dev
       casual          295   11811.591017      4035.866667 6663.816667 22392.595891
       member          187    7116.975847      2484.800000 2918.591667 15382.996204


In [14]:
# 6. Grouped Aggregation for Day of Week
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
clean_df["day_name"] = pd.Categorical(clean_df["day_name"], categories=days, ordered=True)

day_stats = clean_df.groupby(["member_casual", "day_name"], observed=False)["ride_length_min"].agg(
    trips='count',
    mean_min='mean'
).reset_index()

print("\n=== DAY OF WEEK AGGREGATES ===")
print(day_stats.head(14).to_string(index=False))


=== DAY OF WEEK AGGREGATES ===
member_casual  day_name  trips     mean_min
       casual    Monday     22 11413.218182
       casual   Tuesday     23 12898.663768
       casual Wednesday     35  7413.878095
       casual  Thursday     42 18003.119444
       casual    Friday     42 12321.330556
       casual  Saturday     57 10336.847661
       casual    Sunday     74 11004.680180
       member    Monday     18 15964.621296
       member   Tuesday     28  6868.038095
       member Wednesday     24  3546.486111
       member  Thursday     30  3006.928333
       member    Friday     36  6977.623611
       member  Saturday     32  7065.710938
       member    Sunday     19 10451.869298


In [15]:
# Save output for Tableau & Power BI
clean_df.to_csv("cyclistic_cleaned_trips.csv", index=False)